# Principal Component Analysis

- Bui Cao Tri, University Of Information Technology
- Ho Quoc Trung, University Of Information Technology
- Pham Tien, University Of Information Technology
- Phan Cong Thuan, University Of Information Technology


Welcome to the second implementation module of this project: Principal Component Analysis (PCA).

In the previous file, we built Singular Value Decomposition (SVD) from scratch—a powerful matrix factorization algorithm that allowed us to decompose and reconstruct data using the $U \Sigma V^T$ form. While SVD provides the algebraic engine for dimensionality reduction, PCA applies a statistical lens to the same problem.

In this notebook, we will implement PCA to focus on data variance and statistics, demonstrating how it utilizes SVD as its core computational engine to reduce data dimensionality.

#### Import libraries
We rely on the svd module we built in the previous step.

*Note: Ensure svd.py is located in the same directory as this notebook.*

In [ ]:
# Import our custom library "svd.py"
import svd
%load_ext autoreload
%autoreload 2

## Section 1: Data Preprocessing (Centering)

PCA is sensitive to the scale and position of the data. The first and most critical step is Mean Centering.We shift the data so that the center of the dataset aligns with the origin $(0,0)$. This ensures that the first Principal Component describes the direction of maximum variance, rather than just pointing to the data's center of mass.
$$X_{centered} = X - \mu$$

In [ ]:
def get_column_means(A):
    means = []
    n = len(A)
    m = len(A[0])
    for j in range(m):
        sum_col = 0
        for i in range(n):
            sum_col += A[i][j]
        means.append(sum_col / n)
    return means

def centering(A):
    X = svd.copy(A)
    means = get_column_means(A)
    for i in range(len(X)):
        for j in range(len(X[0])):
            X[i][j] -= means[j]
    return X, means

## Section 2: Helper Functions

A simple utility to truncate matrices, allowing us to keep only the top-$k$ most important features.

In [ ]:
def keep_first_k_cols(matrix, k):
    new_matrix = []
    for i in range(len(matrix)):
        row=[]
        limit = min(k, len(matrix[0]))
        for j in range(limit):
            row.append(matrix[i][j])
        new_matrix.append(row)
    return new_matrix

## Section 3: The Core PCA Algorithm

Here we combine our SVD engine with the statistical requirements of PCA.
**The Workflow:**
1. **Center the data** *($X$)*.
2. **Apply SVD** on *$X$*: $X = U \Sigma V^T$.
3. **Extract Components**: The columns of $V$ (or the rows of $V^T$) are the Principal Components (directions of variance).
4. **Project**: We project the data onto these components using matrix multiplication: $Z = X \cdot V$.
5. **Reduce**: We keep only the top $k$ dimensions.

We also calculate the Explained Variance Ratio, which tells us how much information (variance) is preserved after compression, so we know when to update the value of k.

In [ ]:
def PCA(A, k):
    X, means = centering(A)
    U, E, VT = svd.SVD(X)
    V = svd.T(VT)
    Z_full = svd.mul_matr(X, V)
    Z = keep_first_k_cols(Z_full, k)
    V_k = keep_first_k_cols(V, k)

    singular_values = []
    for i in range(len(E)):
        if i < len(E[0]):
            singular_values.append(E[i][i])
    total_variance = sum(val**2 for val in singular_values)
    retained_variance = sum(singular_values[i]**2 for i in range(k))
    explained_ratio = retained_variance / total_variance if total_variance > 0 else 0
    return Z, V_k, means, explained_ratio
        

## Section 4: Inverse Transform & Reconstruction

We can approximate the original data by projecting the reduced data $Z$ back into the original space and adding the mean back.
$$A_{recon} = (Z \cdot V_k^T) + \mu$$

In [ ]:
def inverse_PCA(Z, V_k, means):
    VT_k = svd.T(V_k)
    A_recon = svd.mul_matr(Z, VT_k)

    for i in range(len(A_recon)):
        for j in range(len(A_recon[0])):
            A_recon[i][j] += means[j]
    return A_recon


def calculate_mse(original, reconstructed):
    error = 0
    n = len(original) * len(original[0])
    for i in range(len(original)):
        for j in range(len(original[0])):
            error += (original[i][j] - reconstructed[i][j])**2
    return error / n


## Section 5: Experiment & Evaluation

We generate a random matrix and test if our PCA implementation can effectively reduce dimensions and reconstruct the data.

In [ ]:
if __name__ == "__main__":

    ROWS = 10
    COLS = 30
    A = svd.generate_matrix(ROWS, COLS)
    
    print("--- The original matrix A ---")
    svd.print_matrix(A[:3], "Original...")

    K = 20 # Reduce to K dimension
    print(f"Running PCA to reduce dimensions from {len(A[0])} -> {K}...")
    
    Z, V_k, means, ratio = PCA(A, K)
    
    print(f"\n--- Data after reducing dimensions (Z) ---")
    svd.print_matrix(Z[:3], "Z (Latent Space)")
    
    print(f"Explained Variance Ratio: {ratio * 100:.2f}%")

    print("\n--- (Inverse PCA) ---")
    A_recon = inverse_PCA(Z, V_k, means)
    svd.print_matrix(A_recon[:3], "Reconstructed")

    mse = calculate_mse(A, A_recon)
    print(f"\n===================================")
    print(f"Mean Squared Error (MSE): {mse:.6f}")
    print(f"===================================")
    
    threshold = 1.0
    if mse < threshold:
        print("✅ PCA works well! Reconstruction is reasonably close.")
    else:
        print("⚠️ High error. k might be too small or data is complex.")